In [3]:
import re
import numpy as np
import pandas as pd
import gspread
from google.oauth2.service_account import Credentials

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets.readonly",
    "https://www.googleapis.com/auth/drive.readonly",
]

SERVICE_ACCOUNT_FILE = "service_account.json"
SHEET_ID = "1bRQhejMBoDTXLmT1qGlK1J7Fyalp9htb1valQ6iObDs"
WORKSHEET_NAME = "raw_sales_data"

try:
    creds = Credentials.from_service_account_file(
        SERVICE_ACCOUNT_FILE,
        scopes=SCOPES
    )
    client = gspread.authorize(creds)
    sheet = client.open_by_key(SHEET_ID)
    worksheet = sheet.worksheet(WORKSHEET_NAME)
    records = worksheet.get_all_records()
    df_raw = pd.DataFrame(records)

    print(f"Connected successfully. {len(df_raw)} rows loaded.")
    print("Spreadsheet title:", sheet.title)
    print("Worksheet title:", worksheet.title)
    display(df_raw.head())

except FileNotFoundError:
    raise SystemExit("ERROR: service_account.json not found. Check the file path.")

except gspread.SpreadsheetNotFound:
    raise SystemExit("ERROR: Sheet ID is wrong or the sheet is not shared with the service account.")

except gspread.WorksheetNotFound:
    raise SystemExit(f"ERROR: Tab '{WORKSHEET_NAME}' not found. Check the worksheet name.")

except Exception as e:
    raise SystemExit(f"ERROR: Could not load data — {e}")

Connected successfully. 2877 rows loaded.
Spreadsheet title: Weekly Beauty Report Data
Worksheet title: raw_sales_data


,order_id,date,channel,sku,product_name,category,customer_type,orders,units_sold,gross_revenue,discount_amount,net_revenue,returns
0,ORD10001,2026-02-01,Amazon,FACE002,Skin Blur Primer,Face,New,4,6,5394,761.09,4632.91,0
1,ORD10002,2026-02-01,Flipkart,LIP002,Hydra Gloss Tint,Lips,New,3,4,2396,155.50,2240.5,0
2,ORD10003,2026-02-01,Flipkart,LIP001,Velvet Matte Lipstick,Lips,Returning,4,4,2796,259.83,2536.17,0
3,ORD10004,2026-02-01,Flipkart,SKIN001,Dew Fix Setting Spray,Skincare,Returning,3,4,2996,452.02,2543.98,0
4,ORD10005,2026-02-01,Amazon,FACE003,Cream Blush,Face,New,8,8,5192,417.90,4774.1,0


In [4]:
expected_columns = [
    "order_id",
    "date",
    "channel",
    "sku",
    "product_name",
    "category",
    "customer_type",
    "orders",
    "units_sold",
    "gross_revenue",
    "discount_amount",
    "net_revenue",
    "returns"
]

date_columns = ["date"]

numeric_columns = [
    "orders",
    "units_sold",
    "gross_revenue",
    "discount_amount",
    "net_revenue",
    "returns"
]

text_columns = [
    "order_id",
    "channel",
    "sku",
    "product_name",
    "category",
    "customer_type"
]

print("Expected schema defined.")
print("Expected columns:", expected_columns)

Expected schema defined.
Expected columns: ['order_id', 'date', 'channel', 'sku', 'product_name', 'category', 'customer_type', 'orders', 'units_sold', 'gross_revenue', 'discount_amount', 'net_revenue', 'returns']


In [5]:
try:
    df = df_raw.copy()

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[^\w\s]", "", regex=True)
        .str.replace(r"\s+", "_", regex=True)
    )

    missing_columns = [col for col in expected_columns if col not in df.columns]
    extra_columns = [col for col in df.columns if col not in expected_columns]

    if missing_columns:
        print("WARNING: Missing expected columns:", missing_columns)

    if extra_columns:
        print("INFO: Extra columns found:", extra_columns)

    available_columns = [col for col in expected_columns if col in df.columns]
    df = df[available_columns]

    print("Schema cleaning complete.")
    print("Current columns:", df.columns.tolist())
    print("Shape:", df.shape)
    display(df.head())

except Exception as e:
    raise SystemExit(f"ERROR: Schema cleaning failed — {e}")

Schema cleaning complete.
Current columns: ['order_id', 'date', 'channel', 'sku', 'product_name', 'category', 'customer_type', 'orders', 'units_sold', 'gross_revenue', 'discount_amount', 'net_revenue', 'returns']
Shape: (2877, 13)


,order_id,date,channel,sku,product_name,category,customer_type,orders,units_sold,gross_revenue,discount_amount,net_revenue,returns
0,ORD10001,2026-02-01,Amazon,FACE002,Skin Blur Primer,Face,New,4,6,5394,761.09,4632.91,0
1,ORD10002,2026-02-01,Flipkart,LIP002,Hydra Gloss Tint,Lips,New,3,4,2396,155.50,2240.5,0
2,ORD10003,2026-02-01,Flipkart,LIP001,Velvet Matte Lipstick,Lips,Returning,4,4,2796,259.83,2536.17,0
3,ORD10004,2026-02-01,Flipkart,SKIN001,Dew Fix Setting Spray,Skincare,Returning,3,4,2996,452.02,2543.98,0
4,ORD10005,2026-02-01,Amazon,FACE003,Cream Blush,Face,New,8,8,5192,417.90,4774.1,0


In [6]:
def clean_text_value(x):
    try:
        if pd.isna(x):
            return np.nan
        x = str(x).strip()
        x = re.sub(r"\s+", " ", x)
        return x
    except Exception:
        return np.nan


def clean_channel_name(x):
    try:
        if pd.isna(x):
            return np.nan

        x = str(x).strip().lower()
        x = re.sub(r"\s+", " ", x)

        channel_map = {
            "website": "Website",
            "amazon": "Amazon",
            "flipkart": "Flipkart",
            "instagram": "Instagram Shop",
            "instagram shop": "Instagram Shop",
            "myntra": "Myntra",
            "nykaa": "Nykaa"
        }

        return channel_map.get(x, x.title())

    except Exception:
        return np.nan


try:
    for col in text_columns:
        if col in df.columns:
            df[col] = df[col].apply(clean_text_value)

    if "channel" in df.columns:
        df["channel"] = df["channel"].apply(clean_channel_name)

    print("Channel cleaning complete.")
    print(df["channel"].value_counts(dropna=False))
    display(df[["channel"]].drop_duplicates().sort_values("channel"))

except Exception as e:
    raise SystemExit(f"ERROR: Channel cleaning failed — {e}")

Channel cleaning complete.
channel
Amazon            686
Website           614
Flipkart          467
Nykaa             462
Myntra            362
Instagram Shop    286
Name: count, dtype: int64


,channel
0,Amazon
1,Flipkart
20,Instagram Shop
6,Myntra
9,Nykaa
8,Website


In [7]:
try:
    for col in date_columns:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    for col in numeric_columns:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype(str)
                .str.replace(",", "", regex=False)
                .str.replace("₹", "", regex=False)
                .str.strip()
            )
            df[col] = pd.to_numeric(df[col], errors="coerce")

    print("Numeric and date cleaning complete.\n")

    print("Data types:")
    print(df.dtypes)

    cols_to_check = [col for col in numeric_columns + date_columns if col in df.columns]
    print("\nNulls after conversion:")
    print(df[cols_to_check].isna().sum())

    if "orders" in df.columns:
        print("\nNegative orders count:", (df["orders"] < 0).sum())

    display(df.head())

except Exception as e:
    raise SystemExit(f"ERROR: Numeric/date cleaning failed — {e}")

Numeric and date cleaning complete.

Data types:
order_id                   object
date               datetime64[ns]
channel                    object
sku                        object
product_name               object
category                   object
customer_type              object
orders                      int64
units_sold                  int64
gross_revenue               int64
discount_amount           float64
net_revenue               float64
returns                     int64
dtype: object

Nulls after conversion:
orders             0
units_sold         0
gross_revenue      0
discount_amount    0
net_revenue        3
returns            0
date               1
dtype: int64

Negative orders count: 2


,order_id,date,channel,sku,product_name,category,customer_type,orders,units_sold,gross_revenue,discount_amount,net_revenue,returns
0,ORD10001,2026-02-01,Amazon,FACE002,Skin Blur Primer,Face,New,4,6,5394,761.09,4632.91,0
1,ORD10002,2026-02-01,Flipkart,LIP002,Hydra Gloss Tint,Lips,New,3,4,2396,155.50,2240.50,0
2,ORD10003,2026-02-01,Flipkart,LIP001,Velvet Matte Lipstick,Lips,Returning,4,4,2796,259.83,2536.17,0
3,ORD10004,2026-02-01,Flipkart,SKIN001,Dew Fix Setting Spray,Skincare,Returning,3,4,2996,452.02,2543.98,0
4,ORD10005,2026-02-01,Amazon,FACE003,Cream Blush,Face,New,8,8,5192,417.90,4774.10,0


In [8]:
try:
    df_before_cleaning = df.copy()
    rows_before = len(df)

    # --------------------------------------------------
    # 1. Remove exact duplicate rows
    # --------------------------------------------------
    duplicate_count = df.duplicated().sum()
    df = df.drop_duplicates().reset_index(drop=True)

    # --------------------------------------------------
    # 2. Standardize SKU text first
    # --------------------------------------------------
    if "sku" in df.columns:
        df["sku"] = df["sku"].astype("string").str.strip()
        df["sku"] = df["sku"].replace(
            ["", "nan", "None", "NONE", "null", "NULL"],
            pd.NA
        )

    # --------------------------------------------------
    # 3. Try to fill missing SKU using product_name mapping
    # --------------------------------------------------
    sku_filled_from_product = 0
    sku_filled_from_category = 0

    if {"sku", "product_name"}.issubset(df.columns):
        product_to_sku = (
            df.dropna(subset=["product_name", "sku"])
              .drop_duplicates(subset=["product_name", "sku"])
              .groupby("product_name")["sku"]
              .first()
        )

        missing_sku_mask = df["sku"].isna()
        df.loc[missing_sku_mask, "sku"] = df.loc[missing_sku_mask, "product_name"].map(product_to_sku)
        sku_filled_from_product = int((missing_sku_mask.sum()) - (df["sku"].isna().sum()))

    # --------------------------------------------------
    # 4. Optional fallback: fill SKU from category if category has only one SKU
    # --------------------------------------------------
    if {"sku", "category"}.issubset(df.columns):
        category_sku_counts = (
            df.dropna(subset=["category", "sku"])
              .groupby("category")["sku"]
              .nunique()
        )

        one_sku_categories = category_sku_counts[category_sku_counts == 1].index

        category_to_sku = (
            df[df["category"].isin(one_sku_categories)]
              .dropna(subset=["category", "sku"])
              .groupby("category")["sku"]
              .first()
        )

        before_missing = df["sku"].isna().sum()
        missing_sku_mask = df["sku"].isna()
        df.loc[missing_sku_mask, "sku"] = df.loc[missing_sku_mask, "category"].map(category_to_sku)
        after_missing = df["sku"].isna().sum()
        sku_filled_from_category = int(before_missing - after_missing)

    # --------------------------------------------------
    # 5. Summary
    # --------------------------------------------------
    rows_after = len(df)
    rows_removed = rows_before - rows_after
    remaining_missing_sku = int(df["sku"].isna().sum()) if "sku" in df.columns else None

    print("Cell 6 complete.")
    print("Duplicate rows found:", int(duplicate_count))
    print("Rows removed:", rows_removed)
    print("SKU filled from product_name:", sku_filled_from_product)
    print("SKU filled from category:", sku_filled_from_category)
    print("Remaining missing SKU:", remaining_missing_sku)

    display(df.head())

except Exception as e:
    raise SystemExit(f"ERROR: Duplicate/SKU cleaning failed — {e}")

Cell 6 complete.
Duplicate rows found: 12
Rows removed: 12
SKU filled from product_name: 0
SKU filled from category: 0
Remaining missing SKU: 0


,order_id,date,channel,sku,product_name,category,customer_type,orders,units_sold,gross_revenue,discount_amount,net_revenue,returns
0,ORD10001,2026-02-01,Amazon,FACE002,Skin Blur Primer,Face,New,4,6,5394,761.09,4632.91,0
1,ORD10002,2026-02-01,Flipkart,LIP002,Hydra Gloss Tint,Lips,New,3,4,2396,155.50,2240.50,0
2,ORD10003,2026-02-01,Flipkart,LIP001,Velvet Matte Lipstick,Lips,Returning,4,4,2796,259.83,2536.17,0
3,ORD10004,2026-02-01,Flipkart,SKIN001,Dew Fix Setting Spray,Skincare,Returning,3,4,2996,452.02,2543.98,0
4,ORD10005,2026-02-01,Amazon,FACE003,Cream Blush,Face,New,8,8,5192,417.90,4774.10,0


In [9]:
try:
    print("CLEANING REPORT")
    print("-" * 50)

    print("Rows before cleaning:", len(df_before_cleaning))
    print("Rows after cleaning :", len(df))
    print("Rows removed        :", len(df_before_cleaning) - len(df))

    print("\nDuplicate rows remaining:", int(df.duplicated().sum()))

    print("\nMissing values after cleaning:")
    print(df.isna().sum())

    if "channel" in df.columns:
        print("\nUnique channel values after cleaning:")
        print(sorted(df["channel"].dropna().unique().tolist()))

    if "date" in df.columns:
        print("\nInvalid/missing dates after cleaning:", int(df["date"].isna().sum()))

    numeric_check_cols = [col for col in numeric_columns if col in df.columns]
    if numeric_check_cols:
        print("\nNumeric column summary:")
        print(df[numeric_check_cols].describe())

    if "orders" in df.columns:
        print("\nNegative orders remaining:", int((df["orders"] < 0).sum()))

    print("\nSample cleaned data:")
    display(df.head(10))

except Exception as e:
    raise SystemExit(f"ERROR: Cleaning report failed — {e}")

CLEANING REPORT
--------------------------------------------------
Rows before cleaning: 2877
Rows after cleaning : 2865
Rows removed        : 12

Duplicate rows remaining: 0

Missing values after cleaning:
order_id           0
date               1
channel            0
sku                0
product_name       0
category           0
customer_type      0
orders             0
units_sold         0
gross_revenue      0
discount_amount    0
net_revenue        3
returns            0
dtype: int64

Unique channel values after cleaning:
['Amazon', 'Flipkart', 'Instagram Shop', 'Myntra', 'Nykaa', 'Website']

Invalid/missing dates after cleaning: 1

Numeric column summary:
            orders   units_sold  gross_revenue  discount_amount  net_revenue  \
count  2865.000000  2865.000000    2865.000000      2865.000000  2862.000000   
mean      3.421990     3.969634    2858.491099       291.264101  2567.391041   
std       1.690304     1.945877    1622.856776       209.970245  1474.096094   
min      -1

,order_id,date,channel,sku,product_name,category,customer_type,orders,units_sold,gross_revenue,discount_amount,net_revenue,returns
0,ORD10001,2026-02-01,Amazon,FACE002,Skin Blur Primer,Face,New,4,6,5394,761.09,4632.91,0
1,ORD10002,2026-02-01,Flipkart,LIP002,Hydra Gloss Tint,Lips,New,3,4,2396,155.50,2240.50,0
2,ORD10003,2026-02-01,Flipkart,LIP001,Velvet Matte Lipstick,Lips,Returning,4,4,2796,259.83,2536.17,0
3,ORD10004,2026-02-01,Flipkart,SKIN001,Dew Fix Setting Spray,Skincare,Returning,3,4,2996,452.02,2543.98,0
4,ORD10005,2026-02-01,Amazon,FACE003,Cream Blush,Face,New,8,8,5192,417.90,4774.10,0
5,ORD10006,2026-02-01,Amazon,LIP002,Hydra Gloss Tint,Lips,New,5,6,3594,250.42,3343.58,1
6,ORD10007,2026-02-01,Myntra,LIP001,Velvet Matte Lipstick,Lips,Returning,4,4,2796,242.64,2553.36,0
7,ORD10008,2026-02-01,Myntra,FACE002,Skin Blur Primer,Face,Returning,6,6,5394,628.96,4765.04,0
8,ORD10009,2026-02-01,Website,LIP002,Hydra Gloss Tint,Lips,New,4,5,2995,125.70,2869.30,0
9,ORD10010,2026-02-01,Nykaa,FACE001,Radiance Compact,Face,New,1,1,799,123.94,675.06,0


In [10]:
from pathlib import Path

try:
    # --------------------------------------------
    # 1. Create output folder if it does not exist
    # --------------------------------------------
    output_dir = Path("output")
    output_dir.mkdir(parents=True, exist_ok=True)

    # --------------------------------------------
    # 2. Define output file paths
    # --------------------------------------------
    cleaned_file_path = output_dir / "beauty_sales_cleaned_final.csv"
    review_file_path = output_dir / "beauty_sales_rows_for_review.csv"

    # --------------------------------------------
    # 3. Save cleaned dataset
    # --------------------------------------------
    df.to_csv(cleaned_file_path, index=False)

    # --------------------------------------------
    # 4. Save a review file for rows that may still need attention
    #    (missing sku, missing date, negative orders)
    # --------------------------------------------
    review_conditions = pd.Series(False, index=df.index)

    if "sku" in df.columns:
        review_conditions = review_conditions | df["sku"].isna()

    if "date" in df.columns:
        review_conditions = review_conditions | df["date"].isna()

    if "orders" in df.columns:
        review_conditions = review_conditions | (df["orders"] < 0)

    df_review = df[review_conditions].copy()
    df_review.to_csv(review_file_path, index=False)

    # --------------------------------------------
    # 5. Print success message
    # --------------------------------------------
    print("Phase 3 completed successfully.")
    print("Cleaned file saved at:", cleaned_file_path)
    print("Review file saved at :", review_file_path)
    print("Final cleaned shape  :", df.shape)
    print("Rows for review      :", len(df_review))

except Exception as e:
    raise SystemExit(f"ERROR: Could not save cleaned files — {e}")

Phase 3 completed successfully.
Cleaned file saved at: output\beauty_sales_cleaned_final.csv
Review file saved at : output\beauty_sales_rows_for_review.csv
Final cleaned shape  : (2865, 13)
Rows for review      : 3
